# Day 8: Data Preprocessing & Data Cleaning

**Intern:** Shri Sanjaykumar V  
**Role:** AI/ML Intern  
**Organization:** Linkific  
**Date:** 07 September 2026  

---

## 1. Project Objective & Learning Goals

### 🎯 Learning Objectives:
- Understand why data preprocessing is essential for data science and machine learning.
- Master basic, practical data cleaning techniques in Pandas.
- Detect and impute missing values using context-appropriate methods (median, mode, domain fallback).
- Detect and handle duplicate records.
- Standardize column names to clean, readable conventions.
- Convert columns to correct, memory-efficient data types (`datetime64`, `category`).

### 🔗 Recommended Learning Resources:
- **Documentation:** [Pandas Documentation – Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- **Topics:** Data Cleaning in Python, Data Preprocessing using Pandas, Missing Values in Machine Learning
- **Recommended Channels:** Krish Naik, CampusX, Codebasics

### 💻 Tasks Executed in this Notebook:
1. **Load** raw tabular data and inspect structural properties (`head()`, `shape`, `dtypes`, `info()`).
2. **Identify missing values** per column and overall.
3. **Handle missing values** using context-appropriate techniques (median for skewed numerical, mode for categorical, domain fallback for dates).
4. **Detect and remove duplicate records**.
5. **Rename columns** to standard, consistent Pythonic naming conventions (`snake_case`).
6. **Convert data types** to appropriate formats (`datetime64`, `category`).
7. **Validate** the final cleaned DataFrame.
8. **Save** the cleaned dataset separately as `cleaned_dataset.csv`, preserving the raw `dataset.csv`.

## 2. Libraries Used

We import **Pandas** for tabular manipulation and data cleaning, **NumPy** for numerical operations, and **os** for filesystem path management.

In [1]:
import os
import pandas as pd
import numpy as np

print("Pandas Version :", pd.__version__)
print("NumPy Version  :", np.__version__)
print("All core data cleaning libraries imported successfully!")

Pandas Version : 3.0.3
NumPy Version  : 2.5.0
All core data cleaning libraries imported successfully!


## 3. Load Dataset

We load `dataset.csv`, which contains real employee records from the **City of Houston Public Payroll Dataset** (source: City of Houston Open Data Portal / Pandas Cookbook).
An automatic path resolver ensures the dataset loads properly whether the notebook is executed from the repository root or the `Day-8/` directory.

In [2]:
# Auto-detect dataset path whether running from Day-8 folder or repository root
possible_paths = [
    "dataset.csv",
    os.path.join("Day-8", "dataset.csv"),
    os.path.join("Python", "Day-8", "dataset.csv")
]
data_path = next((p for p in possible_paths if os.path.exists(p)), "dataset.csv")
df = pd.read_csv(data_path)
print(f"Dataset loaded successfully from: {data_path}")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded successfully from: dataset.csv
Dataset Shape: 2000 rows, 10 columns


## 4. Initial Dataset Inspection

Before performing any cleaning, we systematically inspect the raw data to understand its structure:
- `head()`: Displays the first 5 records to get an overview of actual row content.
- `shape`: Shows dimensional bounds `(rows, columns)`.
- `columns`: Lists all column names.
- `dtypes`: Shows the storage data type inferred by Pandas for each column.
- `info()`: Summarizes memory usage, non-null counts, and data types.

In [3]:
# Preview first 5 records
df.head()

Out[0]: 
   UNIQUE_ID               POSITION_TITLE  ...   HIRE_DATE    JOB_DATE
0          0  ASSISTANT DIRECTOR (EX LVL)  ...  2006-06-12  2012-10-13
1          1            LIBRARY ASSISTANT  ...  2000-07-19  2010-09-18
2          2               POLICE OFFICER  ...  2015-02-03  2015-02-03
3          3            ENGINEER/OPERATOR  ...  1982-02-08  1991-05-25
4          4                  ELECTRICIAN  ...  1989-06-19  1994-10-22

[5 rows x 10 columns]


,UNIQUE_ID,POSITION_TITLE,DEPARTMENT,BASE_SALARY,RACE,EMPLOYMENT_TYPE,GENDER,EMPLOYMENT_STATUS,HIRE_DATE,JOB_DATE
0,0,ASSISTANT DIRECTOR (EX LVL),Municipal Courts Department,121862.0,Hispanic/Latino,Full Time,Female,Active,2006-06-12,2012-10-13
1,1,LIBRARY ASSISTANT,Library,26125.0,Hispanic/Latino,Full Time,Female,Active,2000-07-19,2010-09-18
2,2,POLICE OFFICER,Houston Police Department-HPD,45279.0,White,Full Time,Male,Active,2015-02-03,2015-02-03
3,3,ENGINEER/OPERATOR,Houston Fire Department (HFD),63166.0,White,Full Time,Male,Active,1982-02-08,1991-05-25
4,4,ELECTRICIAN,General Services Department,56347.0,White,Full Time,Male,Active,1989-06-19,1994-10-22


In [4]:
print("--- Dataset Dimensions ---")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- Initial Data Types ---")
print(df.dtypes)

print("\n--- DataFrame Info ---")
df.info()

--- Dataset Dimensions ---
Rows    : 2000
Columns : 10

--- Column Names ---
['UNIQUE_ID', 'POSITION_TITLE', 'DEPARTMENT', 'BASE_SALARY', 'RACE', 'EMPLOYMENT_TYPE', 'GENDER', 'EMPLOYMENT_STATUS', 'HIRE_DATE', 'JOB_DATE']

--- Initial Data Types ---
UNIQUE_ID              int64
POSITION_TITLE           str
DEPARTMENT               str
BASE_SALARY          float64
RACE                     str
EMPLOYMENT_TYPE          str
GENDER                   str
EMPLOYMENT_STATUS        str
HIRE_DATE                str
JOB_DATE                 str
dtype: object

--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   UNIQUE_ID          2000 non-null   int64  
 1   POSITION_TITLE     2000 non-null   str    
 2   DEPARTMENT         2000 non-null   str    
 3   BASE_SALARY        1886 non-null   float64
 4   RACE               1965 non-null  

## 5. Missing Value Detection

Missing data (`NaN` / `null`) can distort analytical models and statistical aggregations.
We check for missing values using:
- `df.isnull().sum()`: Counts missing values per column.
- `df.isnull().sum().sum()`: Calculates the grand total of missing entries across the dataset.

In [5]:
print("--- Missing Values per Column ---")
print(df.isnull().sum())

total_missing = df.isnull().sum().sum()
print(f"\nTotal Missing Values in Dataset: {total_missing}")

--- Missing Values per Column ---
UNIQUE_ID              0
POSITION_TITLE         0
DEPARTMENT             0
BASE_SALARY          114
RACE                  35
EMPLOYMENT_TYPE        0
GENDER                 0
EMPLOYMENT_STATUS      0
HIRE_DATE              0
JOB_DATE               3
dtype: int64

Total Missing Values in Dataset: 152


## 6. Missing Value Handling

We apply domain-appropriate imputation techniques rather than dropping rows or blindly applying mean:
- **`BASE_SALARY` (114 missing)**: Salary distributions are typically right-skewed by high-earning management positions. The **median** is outlier-robust and represents the true central tendency better than the mean.
- **`RACE` (35 missing)**: A nominal categorical variable cannot have a mathematical mean or median. We impute with the **mode** (the most frequent category).
- **`JOB_DATE` (3 missing)**: Represents the date an employee assumed their current role. For employees without a recorded job date, their initial **`HIRE_DATE`** serves as the logical administrative proxy.

In [6]:
# 1. Impute numerical BASE_SALARY using median
sal_median = df["BASE_SALARY"].median()
df["BASE_SALARY"] = df["BASE_SALARY"].fillna(sal_median)
print(f"Imputed BASE_SALARY using median: ${sal_median:,.2f}")

# 2. Impute categorical RACE using mode
race_mode = df["RACE"].mode()[0]
df["RACE"] = df["RACE"].fillna(race_mode)
print(f"Imputed RACE using mode: '{race_mode}'")

# 3. Impute missing JOB_DATE using HIRE_DATE
df["JOB_DATE"] = df["JOB_DATE"].fillna(df["HIRE_DATE"])
print("Imputed missing JOB_DATE entries with HIRE_DATE")

# Verification check
print("\n--- Verification: Missing Values After Imputation ---")
print(df.isnull().sum())
print(f"Total Missing Values Remaining: {df.isnull().sum().sum()}")

Imputed BASE_SALARY using median: $54,461.00
Imputed RACE using mode: 'Black or African American'
Imputed missing JOB_DATE entries with HIRE_DATE

--- Verification: Missing Values After Imputation ---
UNIQUE_ID            0
POSITION_TITLE       0
DEPARTMENT           0
BASE_SALARY          0
RACE                 0
EMPLOYMENT_TYPE      0
GENDER               0
EMPLOYMENT_STATUS    0
HIRE_DATE            0
JOB_DATE             0
dtype: int64
Total Missing Values Remaining: 0


## 7. Duplicate Detection

Duplicate records occur due to data ingestion glitches or duplicate exports. If unaddressed, they cause double-counting in calculations.
We detect duplicates using `df.duplicated().sum()`.

In [7]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Records Detected: {duplicate_count}")

Duplicate Records Detected: 0


## 8. Duplicate Removal

We call `df.drop_duplicates()` on the dataset to guarantee idempotency and prevent duplicate records.
We also present a clear, controlled demonstration showing how `drop_duplicates()` operates when repeated rows exist.

In [8]:
# Remove duplicate records if any exist
df = df.drop_duplicates()
print(f"Duplicate records remaining in main dataset: {df.duplicated().sum()}")

# Controlled Demonstration of Duplicate Removal
print("\n--- Controlled Demonstration on Sample Data ---")
demo_df = pd.DataFrame({
    "employee_id": [101, 102, 102, 103],
    "name": ["Aarav", "Diya", "Diya", "Rohan"],
    "department": ["Engineering", "HR", "HR", "Marketing"]
})
print("Sample Data with Duplicate Row (index 2):")
print(demo_df)
print(f"Sample Duplicate Count: {demo_df.duplicated().sum()}")

demo_cleaned = demo_df.drop_duplicates()
print("\nAfter drop_duplicates():")
print(demo_cleaned)
print(f"Sample Duplicate Count After Removal: {demo_cleaned.duplicated().sum()}")

Duplicate records remaining in main dataset: 0

--- Controlled Demonstration on Sample Data ---
Sample Data with Duplicate Row (index 2):
   employee_id   name   department
0          101  Aarav  Engineering
1          102   Diya           HR
2          102   Diya           HR
3          103  Rohan    Marketing
Sample Duplicate Count: 1

After drop_duplicates():
   employee_id   name   department
0          101  Aarav  Engineering
1          102   Diya           HR
3          103  Rohan    Marketing
Sample Duplicate Count After Removal: 0


## 9. Column Renaming

Raw datasets frequently use uppercase abbreviations or inconsistent casing (`UNIQUE_ID`, `BASE_SALARY`).
Renaming columns to clean `snake_case` makes referencing attributes easier in Python code (e.g. `df.base_salary`) and improves readability.
We also rename `UNIQUE_ID` to `employee_id` for clarity.

In [9]:
print("Original Column Names:")
print(df.columns.tolist())

rename_mapping = {
    "UNIQUE_ID": "employee_id",
    "POSITION_TITLE": "position_title",
    "DEPARTMENT": "department",
    "BASE_SALARY": "base_salary",
    "RACE": "race",
    "EMPLOYMENT_TYPE": "employment_type",
    "GENDER": "gender",
    "EMPLOYMENT_STATUS": "employment_status",
    "HIRE_DATE": "hire_date",
    "JOB_DATE": "job_date"
}
df.rename(columns=rename_mapping, inplace=True)

print("\nStandardized snake_case Column Names:")
print(df.columns.tolist())

Original Column Names:
['UNIQUE_ID', 'POSITION_TITLE', 'DEPARTMENT', 'BASE_SALARY', 'RACE', 'EMPLOYMENT_TYPE', 'GENDER', 'EMPLOYMENT_STATUS', 'HIRE_DATE', 'JOB_DATE']

Standardized snake_case Column Names:
['employee_id', 'position_title', 'department', 'base_salary', 'race', 'employment_type', 'gender', 'employment_status', 'hire_date', 'job_date']


## 10. Data Type Inspection

We inspect the column data types after renaming.
Notice that dates (`hire_date`, `job_date`) are stored as generic `object` (string) types, and low-cardinality categorical features (`gender`, `employment_type`) are stored as strings rather than memory-efficient categories.

In [10]:
print("--- Current Data Types ---")
print(df.dtypes)

--- Current Data Types ---
employee_id            int64
position_title           str
department               str
base_salary          float64
race                     str
employment_type          str
gender                   str
employment_status        str
hire_date                str
job_date                 str
dtype: object


## 11. Data Type Conversion

We convert columns to their correct, optimal data types:
- `hire_date` & `job_date`: Converted from `object` (string) to `datetime64[ns]` using `pd.to_datetime()`. This enables temporal calculations (tenure, years of service).
- `gender` & `employment_type`: Converted to `category` using `.astype('category')`. Categorical types optimize memory consumption and speed up grouping operations.

In [11]:
# 1. Convert date strings to datetime64
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")
df["job_date"] = pd.to_datetime(df["job_date"], errors="coerce")

# 2. Convert low-cardinality strings to categorical type
df["gender"] = df["gender"].astype("category")
df["employment_type"] = df["employment_type"].astype("category")

print("--- Data Types After Conversion ---")
print(df.dtypes)

--- Data Types After Conversion ---
employee_id                   int64
position_title                  str
department                      str
base_salary                 float64
race                            str
employment_type            category
gender                     category
employment_status               str
hire_date            datetime64[us]
job_date             datetime64[us]
dtype: object


## 12. Final Dataset Validation

We perform a thorough health check on the final DataFrame to confirm that all missing values have been eliminated, no duplicates exist, column names are clean, and data types are appropriate.

In [12]:
print("=" * 60)
print("               FINAL DATASET HEALTH CHECK                   ")
print("=" * 60)
print(f"Final Shape          : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Total Missing Values : {df.isnull().sum().sum()}")
print(f"Total Duplicate Rows : {df.duplicated().sum()}")
print("\nColumn-wise Missing Values:")
print(df.isnull().sum())
print("\nFirst 5 Records of Cleaned Dataset:")
print(df.head())
print("=" * 60)

               FINAL DATASET HEALTH CHECK                   
Final Shape          : 2000 rows, 10 columns
Total Missing Values : 0
Total Duplicate Rows : 0

Column-wise Missing Values:
employee_id          0
position_title       0
department           0
base_salary          0
race                 0
employment_type      0
gender               0
employment_status    0
hire_date            0
job_date             0
dtype: int64

First 5 Records of Cleaned Dataset:
   employee_id               position_title  ...  hire_date   job_date
0            0  ASSISTANT DIRECTOR (EX LVL)  ... 2006-06-12 2012-10-13
1            1            LIBRARY ASSISTANT  ... 2000-07-19 2010-09-18
2            2               POLICE OFFICER  ... 2015-02-03 2015-02-03
3            3            ENGINEER/OPERATOR  ... 1982-02-08 1991-05-25
4            4                  ELECTRICIAN  ... 1989-06-19 1994-10-22

[5 rows x 10 columns]


## 13. Save Cleaned Dataset

We persist the sanitized DataFrame to `cleaned_dataset.csv` without writing the synthetic integer index (`index=False`).
The original `dataset.csv` is preserved, creating a clear and reproducible data lineage: **Raw Data -> Cleaned Data**.

In [13]:
out_dir = os.path.dirname(data_path) if os.path.dirname(data_path) else "."
out_file = os.path.join(out_dir, "cleaned_dataset.csv")
df.to_csv(out_file, index=False)

print(f"Cleaned dataset saved successfully to: {os.path.abspath(out_file)}")
print(f"File Exists Verification: {os.path.exists(out_file)}")
print(f"Cleaned File Size: {os.path.getsize(out_file):,} bytes")

Cleaned dataset saved successfully to: C:\projects\linkific\internship\Day-8\cleaned_dataset.csv
File Exists Verification: True
Cleaned File Size: 246,951 bytes


## 14. Before vs After Summary

A quantitative before-and-after comparison showing the concrete impact of each data cleaning step.

In [14]:
summary_df = pd.DataFrame({
    "Metric": ["Total Rows", "Total Columns", "Missing Values", "Duplicate Rows", "Datetime Columns", "Category Columns"],
    "Before Cleaning": [2000, 10, 152, 0, 0, 0],
    "After Cleaning": [2000, 10, 0, 0, 2, 2]
})

print("=" * 60)
print("                 BEFORE VS AFTER SUMMARY                    ")
print("=" * 60)
print(summary_df.to_string(index=False))
print("=" * 60)
print("\nKey Transformations Completed:")
print("1. Missing Values: Imputed 114 BASE_SALARY (median), 35 RACE (mode), 3 JOB_DATE (HIRE_DATE).")
print("2. Duplicates: Confirmed 0 duplicate records; verified with drop_duplicates().")
print("3. Column Renaming: Standardized all 10 column names from UPPERCASE to clean snake_case.")
print("4. Data Types: Converted hire_date and job_date to datetime64; gender and employment_type to category.")
print("5. Storage: Maintained raw dataset.csv intact and exported sanitized data to cleaned_dataset.csv.")

                 BEFORE VS AFTER SUMMARY                    
          Metric  Before Cleaning  After Cleaning
      Total Rows             2000            2000
   Total Columns               10              10
  Missing Values              152               0
  Duplicate Rows                0               0
Datetime Columns                0               2
Category Columns                0               2

Key Transformations Completed:
1. Missing Values: Imputed 114 BASE_SALARY (median), 35 RACE (mode), 3 JOB_DATE (HIRE_DATE).
2. Duplicates: Confirmed 0 duplicate records; verified with drop_duplicates().
3. Column Renaming: Standardized all 10 column names from UPPERCASE to clean snake_case.
4. Data Types: Converted hire_date and job_date to datetime64; gender and employment_type to category.
5. Storage: Maintained raw dataset.csv intact and exported sanitized data to cleaned_dataset.csv.


## 15. Conclusion

In this Day 8 Data Preprocessing & Data Cleaning lab:
- Successfully inspected a real 2,000-record public employee dataset.
- Identified and addressed 152 missing values using statistically sound imputation methods.
- Verified duplicate status and demonstrated duplicate removal.
- Standardized all column names to clean, readable `snake_case`.
- Converted date and categorical columns to optimal data types.
- Exported the sanitized data as `cleaned_dataset.csv` ready for subsequent analytical tasks.